In [7]:
"""
IMDB Movie Reviews - Sentiment Analysis Pipeline
==================================================
Covers:
  Task 1-2: EDA (review counts, sentiment distribution, review length)
  Task 3:   Feature Extraction (Bag of Words + TF-IDF)
  Task 4:   Model Building (Naive Bayes, Logistic Regression, SVM)
  Task 5:   Model Evaluation (Accuracy, Precision, Recall, F1, Confusion Matrix)
  Task 6:   Performance Comparison
  Task 7:   Prediction on new/unseen reviews
"""

import re
import time
import joblib
import scipy.sparse
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay
)

# ----------------------------------------------------------------------
# CONFIG
# ----------------------------------------------------------------------
DATA_PATH = "IMDB Dataset.csv"   
MAX_FEATURES = 5000              
RANDOM_STATE = 42
TEST_SIZE = 0.2


# ----------------------------------------------------------------------
# TASK 1-2: LOAD DATA + EDA
# ----------------------------------------------------------------------
def load_and_explore(path):
    df = pd.read_csv(path)

    print("Shape:", df.shape)
    print("\nSentiment value counts:")
    print(df["sentiment"].value_counts())
    print("\nMissing values:")
    print(df.isnull().sum())

    # Sentiment distribution plot
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    counts = df["sentiment"].value_counts()
    axes[0].bar(counts.index, counts.values, color=["#4C72B0", "#DD8452"])
    axes[0].set_title("Sentiment Distribution")
    axes[0].set_xlabel("Sentiment")
    axes[0].set_ylabel("Number of Reviews")
    for i, v in enumerate(counts.values):
        axes[0].text(i, v + 300, str(v), ha="center", fontweight="bold")

    # Review length distribution plot (word count, on raw text)
    review_length = df["review"].apply(lambda x: len(str(x).split()))
    axes[1].hist(review_length, bins=50, color="#55A868", edgecolor="black", alpha=0.8)
    axes[1].set_title("Review Length Distribution (word count)")
    axes[1].set_xlabel("Number of Words")
    axes[1].set_ylabel("Number of Reviews")
    axes[1].axvline(review_length.median(), color="red", linestyle="--",
                     label=f"Median: {review_length.median():.0f}")
    axes[1].legend()

    plt.tight_layout()
    plt.savefig("imdb_eda.png", dpi=150)
    plt.close()
    print("\nSaved plot: imdb_eda.png")

    return df


# ----------------------------------------------------------------------
# TEXT CLEANING
# ----------------------------------------------------------------------
def clean_text(text):
    text = re.sub(r"<.*?>", " ", text)          # remove HTML tags (e.g. <br />)
    text = re.sub(r"[^a-zA-Z\s]", " ", text)     # remove punctuation/numbers
    text = text.lower()                          # lowercase
    text = re.sub(r"\s+", " ", text).strip()     # collapse whitespace
    return text


# ----------------------------------------------------------------------
# TASK 3: FEATURE EXTRACTION (BoW + TF-IDF)
# ----------------------------------------------------------------------
def extract_features(clean_reviews):
    bow_vectorizer = CountVectorizer(max_features=MAX_FEATURES, stop_words="english")
    X_bow = bow_vectorizer.fit_transform(clean_reviews)

    tfidf_vectorizer = TfidfVectorizer(max_features=MAX_FEATURES, stop_words="english")
    X_tfidf = tfidf_vectorizer.fit_transform(clean_reviews)

    print("Bag of Words matrix shape:", X_bow.shape)
    print("TF-IDF matrix shape:      ", X_tfidf.shape)

    return X_bow, bow_vectorizer, X_tfidf, tfidf_vectorizer


# ----------------------------------------------------------------------
# TASK 4: MODEL BUILDING
# ----------------------------------------------------------------------
def train_models(X_train, y_train):
    models = {}

    t0 = time.time()
    nb = MultinomialNB()
    nb.fit(X_train, y_train)
    models["Naive Bayes"] = nb
    print(f"Naive Bayes trained in {time.time() - t0:.2f}s")

    t0 = time.time()
    lr = LogisticRegression(max_iter=1000)
    lr.fit(X_train, y_train)
    models["Logistic Regression"] = lr
    print(f"Logistic Regression trained in {time.time() - t0:.2f}s")

    t0 = time.time()
    svm = LinearSVC()
    svm.fit(X_train, y_train)
    models["SVM"] = svm
    print(f"SVM trained in {time.time() - t0:.2f}s")

    return models


# ----------------------------------------------------------------------
# TASK 5: MODEL EVALUATION
# ----------------------------------------------------------------------
def evaluate_models(models, X_test, y_test):
    results = {}

    fig, axes = plt.subplots(1, 3, figsize=(16, 5))

    for ax, (name, model) in zip(axes, models.items()):
        y_pred = model.predict(X_test)

        acc = accuracy_score(y_test, y_pred)
        prec = precision_score(y_test, y_pred)
        rec = recall_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)
        cm = confusion_matrix(y_test, y_pred)

        results[name] = {"Accuracy": acc, "Precision": prec, "Recall": rec, "F1-score": f1}

        print(f"\n===== {name} =====")
        print(f"Accuracy:  {acc:.4f}")
        print(f"Precision: {prec:.4f}")
        print(f"Recall:    {rec:.4f}")
        print(f"F1-score:  {f1:.4f}")
        print("Confusion Matrix:")
        print(cm)

        disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Negative", "Positive"])
        disp.plot(ax=ax, cmap="Blues", colorbar=False)
        ax.set_title(name)

    plt.tight_layout()
    plt.savefig("confusion_matrices.png", dpi=150)
    plt.close()
    print("\nSaved plot: confusion_matrices.png")

    return results


# ----------------------------------------------------------------------
# TASK 6: PERFORMANCE COMPARISON
# ----------------------------------------------------------------------
def compare_models(results):
    comparison_df = pd.DataFrame(results).T  # rows = models, cols = metrics
    comparison_df = comparison_df[["Accuracy", "Precision", "Recall", "F1-score"]]
    comparison_df = comparison_df.sort_values("F1-score", ascending=False)

    print("\n===== Performance Comparison =====")
    print(comparison_df.round(4))

    best_model_name = comparison_df.index[0]
    print(f"\nBest performing model: {best_model_name}")
    print("Justification: highest F1-score, balancing precision and recall, "
          "while also leading (or tying) on accuracy, precision, and recall individually.")

    return comparison_df, best_model_name


# ----------------------------------------------------------------------
# TASK 7: PREDICTION ON NEW REVIEWS
# ----------------------------------------------------------------------
def predict_new_reviews(model, vectorizer, reviews):
    cleaned = [clean_text(r) for r in reviews]
    X_new = vectorizer.transform(cleaned)
    preds = model.predict(X_new)

    label_map = {1: "Positive", 0: "Negative"}

    print(f"\n{'#':<3}{'Predicted Sentiment':<20}Original Review")
    print("-" * 90)
    for i, (review, pred) in enumerate(zip(reviews, preds), 1):
        print(f"{i:<3}{label_map[pred]:<20}{review}")

    return preds


# ----------------------------------------------------------------------
# MAIN
# ----------------------------------------------------------------------
if __name__ == "__main__":
    # Task 1-2: Load + EDA
    df = load_and_explore(DATA_PATH)

    # Clean text + encode labels
    df["clean_review"] = df["review"].apply(clean_text)
    df["label"] = df["sentiment"].map({"positive": 1, "negative": 0})

    # Task 3: Feature extraction
    X_bow, bow_vectorizer, X_tfidf, tfidf_vectorizer = extract_features(df["clean_review"])

    # Task 4: Train/test split (80/20) + model training (using TF-IDF features)
    X_train, X_test, y_train, y_test = train_test_split(
        X_tfidf, df["label"], test_size=TEST_SIZE,
        random_state=RANDOM_STATE, stratify=df["label"]
    )
    print("\nTrain size:", X_train.shape, " Test size:", X_test.shape)

    models = train_models(X_train, y_train)

    # Task 5: Evaluation
    results = evaluate_models(models, X_test, y_test)

    # Task 6: Comparison
    comparison_df, best_model_name = compare_models(results)
    best_model = models[best_model_name]

    # Task 7: Predict on new/unseen reviews
    new_reviews = [
        "The movie was fantastic with brilliant acting and an emotional story.",
        "I wasted my time watching this movie. The plot was boring and predictable.",
        "The cinematography was beautiful, but the story was average.",
        "An excellent thriller that kept me engaged until the very end.",
        "Poor acting and weak screenplay made this film disappointing."
    ]
    predict_new_reviews(best_model, tfidf_vectorizer, new_reviews)

    # Save everything for reuse
    joblib.dump(models, "models.pkl")
    joblib.dump(tfidf_vectorizer, "tfidf_vectorizer.pkl")
    joblib.dump(bow_vectorizer, "bow_vectorizer.pkl")
    comparison_df.to_csv("model_comparison.csv")
    print("\nSaved: models.pkl, tfidf_vectorizer.pkl, bow_vectorizer.pkl, model_comparison.csv")

Shape: (50000, 2)

Sentiment value counts:
sentiment
positive    25000
negative    25000
Name: count, dtype: int64

Missing values:
review       0
sentiment    0
dtype: int64

Saved plot: imdb_eda.png
Bag of Words matrix shape: (50000, 5000)
TF-IDF matrix shape:       (50000, 5000)

Train size: (40000, 5000)  Test size: (10000, 5000)
Naive Bayes trained in 0.00s
Logistic Regression trained in 0.16s
SVM trained in 0.34s

===== Naive Bayes =====
Accuracy:  0.8525
Precision: 0.8474
Recall:    0.8598
F1-score:  0.8536
Confusion Matrix:
[[4226  774]
 [ 701 4299]]

===== Logistic Regression =====
Accuracy:  0.8898
Precision: 0.8814
Recall:    0.9008
F1-score:  0.8910
Confusion Matrix:
[[4394  606]
 [ 496 4504]]

===== SVM =====
Accuracy:  0.8816
Precision: 0.8768
Recall:    0.8880
F1-score:  0.8824
Confusion Matrix:
[[4376  624]
 [ 560 4440]]

Saved plot: confusion_matrices.png

===== Performance Comparison =====
                     Accuracy  Precision  Recall  F1-score
Logistic Regression 